In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import re
import unicodedata

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

try:
    import geobr
except ImportError as exc:
    raise ImportError("Pacote 'geobr' não encontrado. Instale com: pip install geobr") from exc

if "vscode" in pio.renderers:
    pio.renderers.default = "vscode"
else:
    pio.renderers.default = "notebook_connected"

CSV_MUNICIPIOS = Path("/home/anna/tcc/Tratamento/Municípios/tabela_municipios_regionais.csv")
CSV_RENDA = Path("/home/anna/tcc/Tratamento/Socioeconômico/tabela_renda.csv")
CSV_RACA_COR = Path("/home/anna/tcc/Tratamento/Socioeconômico/tabela_raca_cor.csv")


In [ ]:

def _norm_txt(txt: object) -> str:
    txt = "" if pd.isna(txt) else str(txt)
    txt = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", txt).strip().lower()


def _as_ibge7(val) -> str:
    if pd.isna(val):
        return ""
    s = str(val).strip()
    if re.fullmatch(r"\d+(?:\.0+)?", s):
        s = s.split(".", 1)[0]
    s = re.sub(r"\D", "", s)
    if len(s) > 7:
        s = s[:7]
    return s.zfill(7)


def _fmt_int(n):
    try:
        return f"{int(n):,}".replace(",", ".")
    except Exception:
        return str(n)


for caminho in [CSV_MUNICIPIOS, CSV_RENDA, CSV_RACA_COR]:
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

df_municipios = pd.read_csv(CSV_MUNICIPIOS, dtype={"codigo_ibge": "string"})
df_municipios = df_municipios[["codigo_ibge", "municipio_normalizado"]].copy()
df_municipios["codigo_ibge"] = df_municipios["codigo_ibge"].map(_as_ibge7)

gdf_mg = geobr.read_municipality(code_muni="MG", year=2020, simplified=True).to_crs(4326)
if "code_muni" not in gdf_mg.columns:
    raise KeyError("Não encontrei a coluna 'code_muni' no geobr.")

gdf_mg["codigo_ibge"] = gdf_mg["code_muni"].map(_as_ibge7)
gdf_mg["lon"] = gdf_mg.geometry.representative_point().x
gdf_mg["lat"] = gdf_mg.geometry.representative_point().y

gdf_base = gdf_mg.merge(df_municipios, on="codigo_ibge", how="left")

df_renda = pd.read_csv(CSV_RENDA)
df_renda["codigo_ibge"] = pd.to_numeric(df_renda["Código"], errors="coerce").map(_as_ibge7)
df_renda["Municípios"] = df_renda["Municípios"].astype(str)
colunas_renda = [c for c in df_renda.columns if c not in {"Municípios", "Código", "Ano", "Total", "codigo_ibge"}]

df_raca = pd.read_csv(CSV_RACA_COR)
df_raca["codigo_ibge"] = pd.to_numeric(df_raca["Código"], errors="coerce").map(_as_ibge7)
df_raca["Municípios"] = df_raca["Municípios"].astype(str)
colunas_raca = [c for c in df_raca.columns if c not in {"Municípios", "Código", "Total mulheres", "codigo_ibge"}]

print("Municípios na geometria:", len(gdf_mg))
print("Municípios na renda:", len(df_renda))
print("Municípios na raça/cor:", len(df_raca))
print("Categorias de renda:", colunas_renda)
print("Categorias raça/cor:", colunas_raca)

In [ ]:
def _figura_bolhas_mg(df, categorias, titulo, cor_padrao):
    geojson_mg = json.loads(gdf_mg[["codigo_ibge", "geometry"]].to_json())

    fig = go.Figure()
    fig.add_trace(
        go.Choropleth(
            geojson=geojson_mg,
            locations=gdf_mg["codigo_ibge"],
            z=[1] * len(gdf_mg),
            featureidkey="properties.codigo_ibge",
            colorscale=[[0, "#d9d9d9"], [1, "#d9d9d9"]],
            showscale=False,
            marker_line_color="white",
            marker_line_width=0.3,
            hoverinfo="skip",
            name="Municípios",
        )
    )

    for idx, categoria in enumerate(categorias):
        dados = df[["codigo_ibge", "Municípios", categoria]].copy()
        dados[categoria] = pd.to_numeric(dados[categoria], errors="coerce").fillna(0)
        dados = dados[dados[categoria] > 0].copy()
        if dados.empty:
            continue

        dados = dados.merge(gdf_mg[["codigo_ibge", "lon", "lat"]], on="codigo_ibge", how="left")
        dados = dados.dropna(subset=["lon", "lat"]).copy()
        if dados.empty:
            continue

        max_val = float(dados[categoria].max())
        sizeref = 2.0 * max_val / (45.0 ** 2) if max_val > 0 else 1
        cores = [
            "#2D9CDB", "#D7263D", "#F2994A", "#6FCF97", "#9B51E0",
            "#56CCF2", "#EB5757", "#219653", "#BB6BD9", "#F2C94C",
            "#1B9AAA", "#4F8DF7", "#C0392B", "#7DCEA0",
        ]
        cor = cor_padrao.get(categoria, cores[idx % len(cores)])

        fig.add_trace(
            go.Scattergeo(
                lon=dados["lon"],
                lat=dados["lat"],
                mode="markers",
                name=categoria,
                text=dados["Municípios"],
                customdata=dados[[categoria]],
                hovertemplate=(
                    "Município: %{text}<br>"
                    f"{categoria}: %{{customdata[0]:,.0f}}<extra></extra>"
                ),
                marker=dict(
                    size=dados[categoria],
                    sizemode="area",
                    sizeref=sizeref,
                    color=cor,
                    opacity=0.72,
                    line=dict(color="white", width=0.7),
                ),
            )
        )

    fig.update_layout(
        title=dict(text=titulo, x=0.5),
        paper_bgcolor="white",
        plot_bgcolor="white",
        legend=dict(title="Categoria"),
        margin=dict(l=10, r=10, t=60, b=10),
    )
    fig.update_geos(
        fitbounds="locations",
        visible=False,
        projection_type="mercator",
        showland=False,
        showcoastlines=False,
        showframe=False,
    )
    return fig


cores_renda = {
    "Até 1/4 de salário mínimo": "#1f77b4",
    "Mais de 1/4 a 1/2 salário mínimo": "#17becf",
    "Mais de 1/2 a 1 salário mínimo": "#2ca02c",
    "Mais de 1 a 2 salários mínimos": "#ff7f0e",
    "Mais de 2 a 3 salários mínimos": "#9467bd",
    "Mais de 3 a 5 salários mínimos": "#8c564b",
    "Mais de 5 a 10 salários mínimos": "#e377c2",
    "Mais de 10 a 15 salários mínimos": "#bcbd22",
    "Mais de 15 a 20 salários mínimos": "#d62728",
    "Mais de 20 salários mínimos": "#7f7f7f",
    "Sem rendimento": "#4c78a8",
}

cores_raca = {
    "Parda mulheres": "#9b59b6",
    "Branca mulheres": "#3498db",
    "Amarela mulheres": "#f1c40f",
    "Indígena mulheres": "#2ecc71",
}

fig_renda = _figura_bolhas_mg(
    df_renda,
    colunas_renda,
    "Minas Gerais — distribuição de renda por município",
    cores_renda,
 )
fig_renda.show()

fig_raca = _figura_bolhas_mg(
    df_raca,
    colunas_raca,
    "Minas Gerais — raça/cor por município",
    cores_raca,
 )
fig_raca.show()

In [ ]:
# 4) Mapas de calor por variável (renda e raça/cor) — formato MG, sem bolhas

# ----- AJUSTE DA ESCALA DE RENDA (ALTERE AQUI) -----
ESCALA_MIN = 0
ESCALA_PASSO = 200
ESCALA_MAX = 5000
# -----------------------------------------------

def _plot_mapa_calor_variavel(
    df_origem,
    coluna_valor,
    titulo,
    colorscale="YlOrRd",
    usar_escala_renda=False,
    escala_min=0,
    escala_passo=200,
    escala_max=5000,
):
    geojson_mg = json.loads(gdf_mg[["codigo_ibge", "geometry"]].to_json())

    dados = df_origem[["codigo_ibge", "Municípios", coluna_valor]].copy()
    dados["codigo_ibge"] = dados["codigo_ibge"].map(_as_ibge7)
    dados[coluna_valor] = pd.to_numeric(dados[coluna_valor], errors="coerce").fillna(0)

    base = gdf_mg[["codigo_ibge"]].copy()
    mapa = base.merge(dados, on="codigo_ibge", how="left")
    mapa[coluna_valor] = mapa[coluna_valor].fillna(0)
    mapa["Municípios"] = mapa["Municípios"].fillna("")

    if usar_escala_renda:
        topo = escala_max + 1
        z_plot = mapa[coluna_valor].clip(upper=topo)
        zmin = escala_min
        zmax = topo
        tickvals = list(range(escala_min, escala_max + 1, escala_passo)) + [topo]
        ticktext = [str(v) for v in range(escala_min, escala_max + 1, escala_passo)] + [f">{escala_max}"]
    else:
        z_plot = mapa[coluna_valor]
        zmin = 0
        zmax = float(z_plot.max()) if len(z_plot) else 0.0
        zmax = max(zmax, 1.0)
        tickvals = None
        ticktext = None

    colorbar_cfg = {"title": "Quantidade"}
    if tickvals is not None:
        colorbar_cfg["tickvals"] = tickvals
        colorbar_cfg["ticktext"] = ticktext

    fig = go.Figure(
        go.Choropleth(
            geojson=geojson_mg,
            locations=mapa["codigo_ibge"],
            z=z_plot,
            featureidkey="properties.codigo_ibge",
            colorscale=colorscale,
            zmin=zmin,
            zmax=zmax,
            marker_line_color="white",
            marker_line_width=0.3,
            colorbar=colorbar_cfg,
            customdata=mapa[["Municípios", coluna_valor]],
            hovertemplate="Município: %{customdata[0]}<br>Quantidade: %{customdata[1]:,.0f}<extra></extra>",
        )
    )

    fig.update_layout(
        title=dict(text=titulo, x=0.5),
        margin=dict(l=10, r=10, t=60, b=10),
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    fig.update_geos(
        fitbounds="locations",
        visible=False,
        projection_type="mercator",
        showland=False,
        showcoastlines=False,
        showframe=False,
    )
    fig.show()

print("=== Mapas de calor: Renda ===")
for var_renda in colunas_renda:
    _plot_mapa_calor_variavel(
        df_origem=df_renda,
        coluna_valor=var_renda,
        titulo=f"Mapa de calor (MG) — Renda: {var_renda}",
        colorscale="YlOrRd",
        usar_escala_renda=True,
        escala_min=ESCALA_MIN,
        escala_passo=ESCALA_PASSO,
        escala_max=ESCALA_MAX,
    )

# print("=== Mapas de calor: Raça/Cor ===")
# for var_raca in colunas_raca:
#     _plot_mapa_calor_variavel(
#         df_origem=df_raca,
#         coluna_valor=var_raca,
#         titulo=f"Mapa de calor (MG) — Raça/Cor: {var_raca}",
#         colorscale="PuBuGn",
#     )
